In [ ]:
import pandas as pd
import numpy as np
import os
from datetime import datetime
import janitor
from tqdm.notebook import tqdm
from itertools import islice

FP_POLITICIANS = "output/politicians-edited.csv"
FP_2016RETURNS = "./input/2016-precinct-house.csv"
FP_PARSED_GEOLOC = "../3-parseGeolocation/output/parsed-geoloc.csv"
FP_COUNTY_CENSUS = "../7-merge/election-context-2018.csv"
FP_BOTSCORE = "../4-classifyAccounts/out/bot-classification.csv"
FP_RETURNS2014 = "../7-merge/input/dlatlast/2014_4_0_2/CSV/COUNTY/2014_4_0_2.csv"

### Getting politicians list
    * Gets list of politicians so that I can manually record their gender, black, and incumbent indicators
    * input: input/2016-precinct-house.csv
    * output: output/house-2016.csv.csv
        * edited version with my records is output/politicians-edited.csv

In [ ]:
if 1 == 0:
    columnsToUse = [
        "state",
        "district",
        "county_fips",
        "special",
        "candidate",
        "candidate_full",
        "writein",
        "candidate_party",
        "party",
        "votes",
        "mode",
    ]

    dfOriginal = pd.read_csv(
        "./input/2016-precinct-house.csv",
        usecols=columnsToUse,
        encoding="ISO-8859-1",
    )

    # Drop those without County FIPS
    dfOriginal = dfOriginal[dfOriginal["county_fips"].notnull()]
    # Drop special elections
    dfOriginal = dfOriginal[dfOriginal["special"] == False]
    dfOriginal.drop("special", axis=1, inplace=True)
    # Drop those that are not election days votes
    dfOriginal = dfOriginal[dfOriginal["mode"] != "election day"]
    dfOriginal.drop("mode", axis=1, inplace=True)
    # Drop write-ins
    dfOriginal = dfOriginal[dfOriginal["writein"] == False]
    dfOriginal.drop("writein", axis=1, inplace=True)
    # Set county FIPS to integer
    dfOriginal["county_fips"] = dfOriginal["county_fips"].astype("int")
    dfOriginal["district"] = dfOriginal["district"].astype("str")

    # Drop Party label records
    dfOriginal["candidate"] = dfOriginal["candidate"].str.lower()
    dfOriginal = dfOriginal[~dfOriginal["candidate"].str.contains("party")]

    dfOriginal.drop_duplicates(["candidate"], keep="first", inplace=True)

    dfOriginal.to_csv(os.path.join("output", "politicians.csv"), index=False)

### Read in politician data with gender, black, and incumbent indicators
    * Gets the county-politician level vote data
    * input:
        * input/2016-precinct-house.csv
        * output/politicians-edited.csv

In [ ]:
dfPoliticians = (
    pd.read_csv(FP_POLITICIANS)
    # drop those that should be deleted
    # these are those who no challengers, counties with missing data on main-party candidates and fringe
    # third-party challengers
    .query("delete!=1")
    # impute the incumbents
    .assign(incumbent=lambda df: df["incumbent"].fillna(0).astype("int"))
    # impute politician names (problems mainly with Kentucky, Maine, New Jersey, Texas)
    .assign(manual_name=lambda df: df["manual_name"].fillna(df["candidate"]))
)
dfPoliticians

In [ ]:
politicianNameDict = (
    dfPoliticians[["manual_name", "candidate"]]
    .set_index("candidate")["manual_name"]
    .to_dict()
)
dict(islice(politicianNameDict.items(), 10))

In [ ]:
# 2016 returns 
columnsToUse = [
    "state",
    "state_postal",
    "district",
    "county_name",
    "county_fips",
    "candidate",
    "party",
    "candidate_party",
    "votes",
]
dfOut = (
    pd.read_csv(FP_2016RETURNS,
        usecols=columnsToUse,
        encoding="ISO-8859-1",
    )
    # 3. Drop those nan values under politician (those that should be deleted
    # these are those who no challengers, counties with missing data on main-party candidates and fringe
    # third-party challengers)
    .assign(
        politician=lambda df: df["candidate"].str.lower().map(politicianNameDict)
    )
    .query("politician.notnull()")
    # Aggregate from precinct to county
    .groupby(["state", "state_postal", "district", "county_name", "county_fips", "candidate"])
    .agg({
        "votes": "sum",
        "party": "first",
        "candidate_party": "first",
        "politician": "first"
    })
    .reset_index()
    .remove_columns("candidate")
    # Merge back with dfPoliticians for gender, race, incumbent indicators
    .merge(
        dfPoliticians[["manual_name", "woman", "black", "incumbent"]],
        how="left",
        left_on="politician",
        right_on="manual_name",
    )
    .remove_columns("manual_name")
    # Clean up vars
    .assign(
        woman=lambda df: df["woman"].astype("int"),
        black=lambda df: df["black"].astype("int"),
        county_fips=lambda df: df["county_fips"].astype("int"),
        district=lambda df: df["state"] + " " + df["district"].astype("int").astype("str")
    )
    .assign(
        party=lambda df: df["party"].fillna(df["candidate_party"])
    )
    # Clean state and county names
    .assign(
        state_name=lambda df: df["state"].str.lower().str.strip(),
        state=lambda df: df["state_postal"].str.lower().str.strip(),
        county_name=lambda df: df["county_name"].str.replace("County", "").str.lower().str.strip(),
        county=lambda df: df["county_name"].str.replace("parish", "").str.strip()
    )
    .remove_columns("state_postal")
)
dfOut

#### Check for missing values

In [ ]:
len(dfOut[dfOut["party"].isnull()])

In [ ]:
dfOut[dfOut["county_fips"].isnull()]

In [ ]:
dfOut[dfOut["county_name"].isnull()]

In [ ]:
dfOut[dfOut["district"].isnull()]

In [ ]:
dfOut[dfOut["state"].isnull()]

In [ ]:
dfOut[dfOut["politician"].isnull()]

### Read in tweets and geolocation

**Load in geolocation data**
* input file from ../3-parseGeolocation/output/parsed-geoloc.csv

In [ ]:
# Get human/bot classification
classified_human_accounts = (
    pd.read_csv(FP_BOTSCORE)
    .fillna(0)
    .query("bot_score<=.5")
    ["user_id"]
    .to_list()
)
classified_human_accounts[:4]

In [ ]:
df = (
    pd.read_csv(FP_PARSED_GEOLOC)
    .clean_names()
    .dropna(subset=["primary_county"])
    # Drop classified bot accounts
    .query("author_id in @classified_human_accounts")    
)
print(len(df))
df.head()

**Load in tweets data**
* input files from ../1-downloadTweets/tweets-data/*month*2018.csv

In [ ]:
tweetsFilesList = ['jan2018.csv',
                   'feb2018.csv',
                   'mar2018.csv',                   
                   'apr2018.csv',
                   'may2018.csv', 
                   'jun2018.csv',                   
                   'jul2018.csv',
                   'aug2018.csv',
                   'sep2018.csv',
                   'oct2018.csv',
                   'nov2018.csv']

# get all tweets in 2018 (Jan 2018 to 6 Nov 2018)
dfTweets = pd.DataFrame()

for fileName in tqdm(tweetsFilesList):
    
    _df = pd.read_csv(os.path.join('../1-downloadTweets', 'tweets-data', fileName), 
                      index_col=0)
    dfTweets = pd.concat([dfTweets, _df], ignore_index=True)

print(len(dfTweets))
dfTweets.head()

In [ ]:
dfOutd_tweets_users = (
    df
    .remove_columns(["username"])
    .drop_duplicates(["author_id"])
    # Merge user to tweets
    .merge(dfTweets, on='author_id', how="inner", validate="1:m")
    # ====================================================
    .assign(
        tweet_length=lambda df_: df_['content'].str.len(),
        date=lambda df_: pd.to_datetime(df_['date']),
    )
    # ====================================================
    # Clean up county names
    .assign(
        **{
            'primary_county': lambda df_: (df_['primary_county']
                .str.replace('(city)', '', regex=False)
                .str.replace('city', '', regex=False)
                .str.replace('(', '', regex=False)
                .str.replace(')', '', regex=False)
                .str.strip()
            )
        }
    )
)
dfOutd_tweets_users

In [ ]:
df_county_tweets = (
    dfOutd_tweets_users
    .groupby(['primary_county', 'state_short', 'state_full'])
    .agg({
        'primary_county': 'size',
        'count_hashtags': 'sum',
        'favorites': 'sum',
        'retweets': 'sum',
        'tweet_length': 'sum'
    })
    .rename(columns={'primary_county': 'tweet_count'})
    .reset_index()
    .drop(columns=['state_full'])
    .rename(columns={'primary_county': 'county', 'state_short': 'state'})
    .assign(
        county=lambda df: df["county"].str.replace("parish", "").str.strip()
    )
)
df_county_tweets

### Merge election data and tweets geolocation data

In [ ]:
dfOut = dfOut.merge(df_county_tweets, on=["state", "county"], how="left", validate="m:1")
dfOut

### Get past Presidential election results

** Load Presidential data **

In [ ]:
FP_PRESIDENTIAL_RETURNS = "./input/countypres_2000-2016.csv"

In [ ]:
# Read in and prepare presidential data
dfPresidential = (
    pd.read_csv(FP_PRESIDENTIAL_RETURNS)
    .query("year >= 2004")
    .query("FIPS.notnull()")
    .query("candidatevotes.notnull()")
    .query("party.notnull()")
    .astype({"FIPS": "int", "candidatevotes": "int"})
    .drop(columns=["totalvotes", "office", "version", "candidate", "state_po"])
)
dfPresidential

** Generate 2004 presidential vote **

In [ ]:
# Get county-level two-party vote totals
df2004TotalVotes = (
    dfPresidential[dfPresidential["year"] == 2004]
    .groupby("FIPS")["candidatevotes"]
    .sum()
    .reset_index()
    .rename(columns={"FIPS": "county_fips", "candidatevotes": "totalPresVotes2004"})
)
df2004TotalVotes

In [ ]:
dfPresDem2004 = (
    dfPresidential
    .query("year == 2004 and party == 'democrat'")
    [["FIPS", "candidatevotes"]]
    .rename(columns={"FIPS": "county_fips", "candidatevotes": "demPresVotes2004"})
)

dfPresRep2004 = (
    dfPresidential
    .query("year == 2004 and party == 'republican'")
    [["FIPS", "candidatevotes"]]
    .rename(columns={"FIPS": "county_fips", "candidatevotes": "repPresVotes2004"})
)
dfPresRep2004

In [ ]:
dfOut = dfOut.merge(dfPresDem2004, how="left", on="county_fips")
dfOut = dfOut.merge(dfPresRep2004, how="left", on="county_fips")
dfOut = dfOut.merge(df2004TotalVotes, how="left", on="county_fips")
dfOut.head()

** Generate 2008 presidential vote **

In [ ]:
# Get county-level two-party vote totals
df2008TotalVotes = (
    dfPresidential
    .query("year == 2008")
    .groupby("FIPS")["candidatevotes"]
    .sum()
    .reset_index()
    .rename(columns={"FIPS": "county_fips", "candidatevotes": "totalPresVotes2008"})
)
df2008TotalVotes

In [ ]:
dfPresDem2008 = (
    dfPresidential
    .query("year == 2008 and party == 'democrat'")
    [["FIPS", "candidatevotes"]]
    .rename(columns={"FIPS": "county_fips", "candidatevotes": "demPresVotes2008"})
)

dfPresRep2008 = (
    dfPresidential
    .query("year == 2008 and party == 'republican'")
    [["FIPS", "candidatevotes"]]
    .rename(columns={"FIPS": "county_fips", "candidatevotes": "repPresVotes2008"})
)
dfPresRep2008

In [ ]:
dfOut = dfOut.merge(dfPresDem2008, how="left", on="county_fips")
dfOut = dfOut.merge(dfPresRep2008, how="left", on="county_fips")
dfOut = dfOut.merge(df2008TotalVotes, how="left", on="county_fips")
dfOut

** Generate 2012 presidential vote **

In [ ]:
# Get county-level two-party vote totals
df2012TotalVotes = (
    dfPresidential
    .query("year == 2012")
    .groupby("FIPS")["candidatevotes"]
    .sum()
    .reset_index()
    .rename(columns={"FIPS": "county_fips", "candidatevotes": "totalPresVotes2012"})
)
df2012TotalVotes

In [ ]:
dfPresDem2012 = (
    dfPresidential
    .query("year == 2012 and party == 'democrat'")
    [["FIPS", "candidatevotes"]]
    .rename(columns={"FIPS": "county_fips", "candidatevotes": "demPresVotes2012"})
)

dfPresRep2012 = (
    dfPresidential
    .query("year == 2012 and party == 'republican'")
    [["FIPS", "candidatevotes"]]
    .rename(columns={"FIPS": "county_fips", "candidatevotes": "repPresVotes2012"})
)
dfPresRep2012

In [ ]:
dfOut = dfOut.merge(dfPresDem2012, how="left", on="county_fips")
dfOut = dfOut.merge(dfPresRep2012, how="left", on="county_fips")
dfOut = dfOut.merge(df2012TotalVotes, how="left", on="county_fips")
dfOut

** Generate 2016 presidential vote **

In [ ]:
# Get county-level two-party vote totals
df2016TotalVotes = (
    dfPresidential
    .query("year == 2016")
    .groupby("FIPS")["candidatevotes"]
    .sum()
    .reset_index()
    .rename(columns={"FIPS": "county_fips", "candidatevotes": "totalPresVotes2016"})
)
df2016TotalVotes

In [ ]:
dfPresDem2016 = (
    dfPresidential
    .query("year == 2016 and party == 'democrat'")
    [["FIPS", "candidatevotes"]]
    .rename(columns={"FIPS": "county_fips", "candidatevotes": "demPresVotes2016"})
)

dfPresRep2016 = (
    dfPresidential
    .query("year == 2016 and party == 'republican'")
    [["FIPS", "candidatevotes"]]
    .rename(columns={"FIPS": "county_fips", "candidatevotes": "repPresVotes2016"})
)
dfPresRep2016

In [ ]:
dfOut = dfOut.merge(dfPresDem2016, how="left", on="county_fips")
dfOut = dfOut.merge(dfPresRep2016, how="left", on="county_fips")
dfOut = dfOut.merge(df2016TotalVotes, how="left", on="county_fips")
dfOut

### Get county-level demographics
    * input: ../5-mergeForStata/election-context-2018.csv

In [ ]:
colToUse = [
    "fips",
    "repgov14",
    "demgov14",
    "othergov14",
    "total_population",
    "cvap",
    "white_pct",
    "black_pct",
    "hispanic_pct",
    "nonwhite_pct",
    "foreignborn_pct",
    "female_pct",
    "age29andunder_pct",
    "age65andolder_pct",
    "median_hh_inc",
    "clf_unemploy_pct",
    "lesshs_pct",
    "lesscollege_pct",
    "lesshs_whites_pct",
    "lesscollege_whites_pct",
    "rural_pct",
    "ruralurban_cc",
]

dfCountyDemo = (
    pd.read_csv(FP_COUNTY_CENSUS, usecols=colToUse)
    .rename_column("fips", "county_fips")
)
dfCountyDemo

In [ ]:
dfOut = (
    dfOut.merge(dfCountyDemo, how="left", on="county_fips", validate="m:1")
    # Generate party indicators
    .assign(
        party=lambda df: df["party"].str.lower(),
        democrat=lambda df: np.where(
            df["party"].str.lower().str.contains("democratic|dem", na=False), 1, 0
        ),
        republican=lambda df: np.where(
            df["party"].str.lower().str.contains("republican|rep", na=False), 1, 0
        )
    )
    .assign(
        independent=lambda df: np.where(
            (df["democrat"] + df["republican"]) == 0, 1, 0
        )
    )    
)
dfOut

In [ ]:
# Generate openseat indicator
dfOpenSeat = (
    dfOut
    .groupby("district")["incumbent"]
    .sum()
    .reset_index()
    .assign(
        openseat=lambda df: np.where(df["incumbent"] == 0, 1, 0)
    )
    .drop("incumbent", axis=1)
)
dfOpenSeat

In [ ]:
dfOut = dfOut.merge(dfOpenSeat, how="left", on="district", validate="m:1")
dfOut

### Generating additional indicators and variables

** Total county votes **

In [ ]:
df_county_votes = (
    dfOut
    .groupby(["state", "county"])["votes"]
    .sum()
    .reset_index()
    .rename(columns={"votes": "countyvotes"})
)
df_county_votes

In [ ]:
# Merge back
dfOut = dfOut.merge(df_county_votes, on=["state", "county"], how="left", validate="m:1")
dfOut

In [ ]:
# Create district-county votes aggregation
df_district_county_votes = (
    dfOut
    .groupby(["district", "county"])["votes"]
    .sum()
    .reset_index()
    .rename(columns={"votes": "districtcountyvotes"})
)

# Merge back
dfOut = dfOut.merge(df_district_county_votes, on=["district", "county"], how="left", validate="m:1")
dfOut

** No main challenger **

In [ ]:
# Create no main challenger indicator
df_main_challenger = (
    dfOut
    .groupby("district")[["democrat", "republican"]]
    .sum()
    .reset_index()
    .assign(
        placeholder=lambda df: df["democrat"] + df["republican"],
        no_main_challenger=lambda df: (df["placeholder"] == 1).astype("int")
    )
    .drop(["democrat", "republican", "placeholder"], axis=1)
)
df_main_challenger

In [ ]:
# Merge back
dfOut = dfOut.merge(df_main_challenger, on="district", how="left", validate="m:1")

** At least 1 woman candidate **

In [ ]:
# Create at least 1 woman indicator
df_woman = (
    dfOut
    .groupby("district")["woman"]
    .sum()
    .reset_index()
    .assign(
        atleast1woman=lambda df: (df["woman"] > 0).astype("int")
    )
    .drop("woman", axis=1)
)
df_woman

In [ ]:
# Merge back
dfOut = dfOut.merge(df_woman, on="district", how="left", validate="m:1")

** Which districts have at least one main candidate is a woman **

In [ ]:
# Create at least 1 woman main party candidate indicator
df_woman_main = (
    dfOut
    .query("no_main_challenger == 0")  # Drop no main challenger districts
    .query("republican == 1 | democrat == 1")  # Keep only main party candidates
    .assign(
        womanmain=lambda df: (
            (df["republican"] * df["woman"] + df["democrat"] * df["woman"]) == 1
        ).astype("int")
    )
    .groupby("district")["womanmain"]
    .sum()
    .reset_index()
    .assign(
        atleast1womanmain=lambda df: (df["womanmain"] > 0).astype("int")
    )
    .drop("womanmain", axis=1)
)
df_woman_main

In [ ]:
# Merge back
dfOut = dfOut.merge(df_woman_main, on="district", how="left")
dfOut["atleast1womanmain"] = dfOut["atleast1womanmain"].fillna(0).astype("int")

### Get county Republican votes

In [ ]:
# Create Republican county votes
df_rep_county_votes = (
    dfOut
    .query("republican == 1")
    .groupby(["district", "county"])["votes"]
    .sum()
    .reset_index()
    .rename(columns={"votes": "repcountyvotes"})
)
df_rep_county_votes

In [ ]:
# Merge back
dfOut = dfOut.merge(df_rep_county_votes, on=["district", "county"], how="left", validate="m:1")

### Get county Democratic votes

In [ ]:
# Create Democrat county votes
df_dem_county_votes = (
    dfOut
    .query("democrat == 1")
    .groupby(["district", "county"])["votes"]
    .sum()
    .reset_index()
    .rename(columns={"votes": "demcountyvotes"})
)
df_dem_county_votes

In [ ]:
# Merge back
dfOut = dfOut.merge(df_dem_county_votes, on=["district", "county"], how="left", validate="m:1")

### Get total two-party county votes

In [ ]:
# Create two-party county votes
df_two_party_county_votes = (
    dfOut
    .query("republican == 1 | democrat == 1")
    .groupby(["district", "county"])["votes"]
    .sum()
    .reset_index()
    .rename(columns={"votes": "twoPtyCountyVotes"})
)
df_two_party_county_votes

In [ ]:
## 2014 turnout# Merge back
dfOut = dfOut.merge(df_two_party_county_votes, on=["district", "county"], how="left", validate="m:1")

### 2014 turnout

In [ ]:
df_house2014 = (
    pd.read_csv(FP_RETURNS2014, skiprows=1)
    .filter(["fips", "totalvote", "vote1", "vote2"])
    .rename_column("vote1", "repHouse14")
    .rename_column("vote2", "demHouse14")
    .rename_column("totalvote", "totalHouse14")
    .rename_column("fips", "county_fips")
)
df_house2014

In [ ]:
dfOut = dfOut.merge(df_house2014, on="county_fips", how="left", validate="m:1")
dfOut

### Candidacy

In [ ]:
_df = (
    dfOut
    .filter(['district', 'woman', 'incumbent'])
    .query("incumbent==0")
    .remove_columns("incumbent")
    .groupby("district")
    .sum()
    .reset_index()
    .assign(atLeast1WomanChallenger=lambda df_: np.where(df_['woman'] > 0, 1, 0 ))
    .remove_columns("woman")
)
dfOut = (
    dfOut
    .merge(_df, how="left", on="district", validate="m:1")
    .fillna({'atLeast1WomanChallenger': 0})
)
_df

In [ ]:
_df = (
    dfOut
    .filter(['district', 'woman', 'democrat', 'incumbent'])
    .query("incumbent==0")
    .query('democrat==1')
    .remove_columns(["incumbent", "democrat"])
    .groupby("district")
    .sum()
    .reset_index()
    .assign(atLeast1DemWomanChallenger=lambda df_: np.where(df_['woman'] > 0, 1, 0 ))
    .remove_columns("woman")
)
dfOut = (
    dfOut
    .merge(_df, how="left", on="district", validate="m:1")
    .fillna({'atLeast1DemWomanChallenger': 0})
)
assert (dfOut["atLeast1DemWomanChallenger"] <= dfOut["atLeast1WomanChallenger"]).all()
_df

In [ ]:
_df = (
    dfOut
    .filter(['district', 'woman', 'republican', 'incumbent'])
    .query("incumbent==0")
    .query('republican==1')
    .remove_columns(["incumbent", "republican"])
    .groupby("district")
    .sum()
    .reset_index()
    .assign(atLeast1RepWomanChallenger=lambda df_: np.where(df_['woman'] > 0, 1, 0 ))
    .remove_columns("woman")
)
dfOut = (
    dfOut
    .merge(_df, how="left", on="district", validate="m:1")
    .fillna({'atLeast1RepWomanChallenger': 0})
)
assert (dfOut["atLeast1RepWomanChallenger"] <= dfOut["atLeast1WomanChallenger"]).all()
_df

In [ ]:
_df = (
    dfOut
    .filter(['district', 'incumbent', 'woman'])
    .query("incumbent==1")
    .groupby('district')['woman']
    .sum()
    .reset_index()
    .assign(manIncumbent=lambda df_: (df_['woman'] == 0).astype(int))
    .remove_columns("woman")
)
dfOut = (
    dfOut
    .merge(_df, how="left", on="district", validate="m:1")
    .fillna({'manIncumbent': 0})
)
_df

In [ ]:
_df = (
    dfOut
    .filter(['district', 'incumbent', 'republican'])
    .query("incumbent==1")
    .groupby('district')['republican']
    .sum()
    .reset_index()
    .assign(repIncumbent=lambda df_: (df_['republican'] > 0).astype(int))
    .remove_columns("republican")
)
dfOut = (
    dfOut
    .merge(_df, how="left", on="district", validate="m:1")
    .fillna({'repIncumbent': 0})
)
_df

In [ ]:
_df = (
    dfOut
    .filter(['district', 'incumbent', 'democrat'])  # or whatever your Democrat column is called
    .query("incumbent==1")
    .groupby('district')['democrat']
    .sum()
    .reset_index()
    .assign(demIncumbent=lambda df_: (df_['democrat'] > 0).astype(int))
    .remove_columns("democrat")
)
dfOut = (
    dfOut
    .merge(_df, how="left", on="district", validate="m:1")
    .fillna({'demIncumbent': 0})
)

_df

In [ ]:
dfOut.to_csv(os.path.join("output", "house-2016-stata.csv"), index=False)